# 01 — เปรียบเทียบโมเดลสำหรับจำแนกส่วนผสม

เทียบ **DenseNet121, EfficientNetB0, MobileNetV2, InceptionV3, ResNet50** ภายใต้เงื่อนไขเดียวกัน (ข้อมูล split เดียวกัน, seed, ขนาดรูป 224px, augmentation, loss, epoch, learning rate)

ผลลัพธ์ใช้เลือก backbone ไปเทรนจริงใน `02_train_final.ipynb`

**ก่อนรัน:** รันไทม์ → เปลี่ยนประเภทรันไทม์ → **T4 GPU** แล้วแก้ `IMAGE_ROOT` ใน cell แรก จากนั้น รันไทม์ → เรียกใช้ทั้งหมด

## 1. ตั้งค่า

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys

PROJECT    = '/content/drive/MyDrive/Colab Notebooks/ingredient_model'
IMAGE_ROOT = '/content/drive/MyDrive/Colab Notebooks/image_ingredient_dataset'
DATA_DIR   = '/content/data_prepared'
COMPARE_DIR = f'{PROJECT}/runs/compare'
os.chdir(PROJECT)
print(os.listdir(PROJECT))
!nvidia-smi -L

['classes_set1.json', 'labels_final_set2_1584.json', 'labels_final_set1_1669.json', 'labels_final_set1_1669_fixed.json', 'labels_final_set2_1584_fixed.json', 'runs']
GPU 0: Tesla T4 (UUID: GPU-7f9a74e5-5194-7c8a-3b27-911c7f91dca4)


In [ ]:
import json
from pathlib import Path
from collections import defaultdict

image_root = Path(IMAGE_ROOT)
label_files = ["labels_final_set1_1669.json", "labels_final_set2_1584.json"]

key = lambda p: Path(p).stem.lower().replace(" ", "")
index = defaultdict(list)
for p in image_root.rglob("*"):
    if p.is_file():
        index[key(p)].append(p.relative_to(image_root).as_posix())

for f in label_files:
    data = json.load(open(f, encoding="utf-8"))
    fixed, dropped = [], []

    for row in data:
        img = row["image"]
        if (image_root / img).exists():
            fixed.append(row)
            continue

        found = index.get(key(img), [])
        if len(found) != 1:          # ไม่เจอ หรือเจอหลายไฟล์ → ตัดทิ้ง
            dropped.append(img)
            continue

        new = found[0]
        old_folder = Path(img).parent.name
        new_folder = Path(new).parent.name
        row["image"] = new
        fixed.append(row)

        if old_folder != new_folder:  # ย้ายโฟลเดอร์ → ต้องเช็ค label
            on = [k for k, v in row["labels"].items() if v]
            print(f"⚠️ เช็ค label: {new}\n   (เดิมอยู่ใน {old_folder}) labels = {on}")

    out = f.replace(".json", "_fixed.json")
    json.dump(fixed, open(out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"{f}: {len(data)} → {len(fixed)} แถว (ตัดทิ้ง {dropped}) → {out}\n")

labels_final_set1_1669.json: 1669 → 1669 แถว (ตัดทิ้ง []) → labels_final_set1_1669_fixed.json

⚠️ เช็ค label: Set 2/panang/panangbeef/panangbeef(110).jpg
   (เดิมอยู่ใน panangchicken) labels = ['chicken', 'chili', 'kaffir_lime_leaf', 'green_eggplant']
⚠️ เช็ค label: Set 2/tom yum/tom yum pla/tom yum pla (6).jpg
   (เดิมอยู่ใน tom yum kung nam sai) labels = ['fish', 'chili', 'coriander', 'kaffir_lime_leaf']
⚠️ เช็ค label: Set 2/tom yum/tom yum pla/tom yum pla (7).jpg
   (เดิมอยู่ใน tom yum kung nam sai) labels = ['fish', 'chili', 'lime', 'kaffir_lime_leaf']
⚠️ เช็ค label: Set 2/tom yum/tom yum pla/tom yum pla (8).jpg
   (เดิมอยู่ใน tom yum kung nam sai) labels = ['fish', 'chili', 'tomato', 'coriander', 'kaffir_lime_leaf']
⚠️ เช็ค label: Set 2/tom yum/tom yum pla/tom yum pla (9).jpg
   (เดิมอยู่ใน tom yum kung nam sai) labels = ['fish', 'chili', 'mushroom', 'coriander', 'kaffir_lime_leaf']
⚠️ เช็ค label: Set 2/tom yum/tom yum pla/tom yum pla (10).jpg
   (เดิมอยู่ใน tom yum kung nam sai) 

In [ ]:
MODELS      = ["densenet121", "efficientnetb0", "mobilenetv2", "inceptionv3", "resnet50"]
IMG_SIZE    = 224
BATCH       = 32
EPOCHS_HEAD = 5
EPOCHS_FT   = 15
SEED        = 42

## โค้ดเตรียมข้อมูล
รวม Set 1 + Set 2 → จัดการรูปที่ label ซ้ำ → ตัดส่วนผสมที่มี < 30 รูป → แปลงรูปเป็น JPEG → แบ่ง train/val/test 70/15/15 ตามเมนูย่อย

In [ ]:
import csv
import hashlib
import json
import random
from collections import defaultdict
from pathlib import Path

from PIL import Image, ImageOps

# อ่านไฟล์ JSON
def read_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


# แยกชื่อเมนูและเมนูย่อย
def get_menu(image_path):
    parts = image_path.replace("\\", "/").split("/")

    if len(parts) >= 3:
        menu = parts[1]
        variant = parts[-2]

    else:
        menu = "unknown"
        variant = "unknown"

    return menu, variant


# คำนวณ Hash ของรูปภาพ
def get_hash(image):
    image = image.convert("L")
    image = image.resize((17, 16), Image.LANCZOS)

    pixels = list(image.getdata())
    image_hash = 0

    for row in range(16):
        for col in range(16):
            left = pixels[row * 17 + col]
            right = pixels[row * 17 + col + 1]

            image_hash = image_hash << 1

            if left > right:
                image_hash += 1

    return image_hash


# รวมไฟล์ Label
def combine_labels(label_files, all_classes):
    images = {}

    for file_index, label_file in enumerate(label_files):
        data = read_json(label_file)

        for row in data:
            image_path = row["image"]

            if image_path not in images:
                images[image_path] = []

            images[image_path].append(
                (file_index, row["id"], row["labels"])
            )

    rows = []

    for image_path, items in images.items():
        items.sort(key=lambda item: (item[0], item[1]))

        # ใช้ Label จากอันล่าสุด
        labels = items[-1][2]

        new_row = {
            "image": image_path,
            "labels": {}
        }

        for name in all_classes:
            new_row["labels"][name] = int(labels.get(name, 0))

        rows.append(new_row)

    return rows


# prepare image

import time

def prepare_images(rows, image_root, output):
    cache_dir = output / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)

    usable_images = []
    missing_count = 0
    start_time = time.time()

    for i, row in enumerate(rows, start=1):
        source = image_root / row["image"]

        file_name = hashlib.sha1(
            row["image"].encode()
        ).hexdigest()[:16] + ".jpg"

        target = cache_dir / file_name

        try:
            if target.exists():
                with Image.open(target) as image:
                    image_hash = get_hash(image)

            else:
                with Image.open(source) as image:
                    image = ImageOps.exif_transpose(image)
                    image = image.convert("RGB")
                    image.thumbnail((640, 640), Image.LANCZOS)
                    image.save(target, "JPEG", quality=92)

                    # คำนวณ Hash จากรูปที่เปิดอยู่แล้ว
                    image_hash = get_hash(image)

            row["hash"] = image_hash
            row["cache_path"] = str(target.relative_to(output))
            row["dish"], row["variant"] = get_menu(row["image"])

            usable_images.append(row)

        except Exception as e:
            missing_count += 1
            print(f"ข้ามรูป: {row['image']} ({e})")

        # แสดงความคืบหน้าทุก 100 รูป
        if i % 100 == 0 or i == len(rows):
            elapsed = time.time() - start_time
            print(f"ประมวลผล {i}/{len(rows)} รูป | "
                  f"ใช้เวลา {elapsed:.1f} วินาที")

    print("จำนวน usable image:", len(usable_images))
    print("จำนวน missing image:", missing_count)

    return usable_images


# เลือกส่วนผสมที่มีข้อมูลพอ
def select_classes(rows, all_classes, min_pos):
    counts = {}

    for name in all_classes:
        counts[name] = 0

    for row in rows:
        for name in all_classes:
            if row["labels"][name] == 1:
                counts[name] += 1

    classes = []

    for name in all_classes:
        if counts[name] >= min_pos:
            classes.append(name)

    return classes


# จัดกลุ่มรูปที่มี Hash ตรงกัน
def group_images(rows):
    hash_groups = {}
    groups = defaultdict(list)

    for row in rows:
        image_hash = row["hash"]

        if image_hash == 0:
            group_name = row["image"]
        else:
            if image_hash not in hash_groups:
                hash_groups[image_hash] = row["image"]

            group_name = hash_groups[image_hash]

        groups[group_name].append(row)

    return groups


# แบ่ง Train, Validation และ Test
def split_dataset(groups):
    random.seed(42)

    groups_by_variant = defaultdict(list)

    for group in groups.values():
        variant = group[0]["variant"]
        groups_by_variant[variant].append(group)

    for variant_groups in groups_by_variant.values():
        random.shuffle(variant_groups)

        total = len(variant_groups)

        if total >= 3:
            test_count = max(1, round(total * 0.15))
            val_count = max(1, round(total * 0.15))

        else:
            test_count = 0
            val_count = 0

        for index, group in enumerate(variant_groups):
            if index < test_count:
                split = "test"

            elif index < test_count + val_count:
                split = "val"

            else:
                split = "train"

            for row in group:
                row["split"] = split


# บันทึก Dataset เป็น CSV
def save_dataset(rows, classes, output):
    csv_file = output / "dataset.csv"

    with open(csv_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        writer.writerow([
            "cache_path",
            "image",
            "dish",
            "variant",
            "split"
        ] + classes)

        rows.sort(key=lambda row: row["image"])

        for row in rows:
            values = [
                row["cache_path"],
                row["image"],
                row["dish"],
                row["variant"],
                row["split"]
            ]

            for name in classes:
                values.append(row["labels"][name])

            writer.writerow(values)

    return csv_file


# ขั้นตอนหลัก
def prepare_dataset(label_files, classes_file, image_root, out="data_prepared", min_pos=30):
    output = Path(out)
    output.mkdir(parents=True, exist_ok=True)

    # 1. อ่านรายชื่อส่วนผสม
    all_classes = read_json(classes_file)

    # 2. รวม Label
    rows = combine_labels(label_files, all_classes)
    print("จำนวนรูปหลังรวม Label:", len(rows))

    # 3. เตรียมรูปภาพ
    rows = prepare_images(rows, Path(image_root), output)

    if len(rows) == 0:
        raise SystemExit("ไม่พบรูปภาพ กรุณาตรวจสอบ image_root")

    # 4. เลือกส่วนผสมที่มีข้อมูลเพียงพอ
    classes = select_classes(rows, all_classes, min_pos)
    print("จำนวนส่วนผสมที่ใช้:", len(classes))

    # บันทึกรายชื่อส่วนผสม (ขั้นเทรนต้องใช้ไฟล์นี้)
    with open(output / "classes.json", "w", encoding="utf-8") as file:
        json.dump(classes, file, indent=2, ensure_ascii=False)

    # 5. จัดกลุ่มรูปที่มี Hash ตรงกัน
    groups = group_images(rows)

    # 6. แบ่ง Dataset
    split_dataset(groups)

    # 7. บันทึก CSV
    csv_file = save_dataset(rows, classes, output)

    print("\nเตรียม Dataset เสร็จแล้ว")
    print("ไฟล์ข้อมูล:", csv_file)

In [ ]:
!rm -rf {DATA_DIR}

prepare_dataset(
       label_files=["labels_final_set1_1669_fixed.json",
                    "labels_final_set2_1584_fixed.json"],
       classes_file="classes_set1.json",
       image_root=IMAGE_ROOT,
       out=DATA_DIR,
)

จำนวนรูปหลังรวม Label: 2965


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


ประมวลผล 100/2965 รูป | ใช้เวลา 8.9 วินาที
ประมวลผล 200/2965 รูป | ใช้เวลา 68.6 วินาที
ประมวลผล 300/2965 รูป | ใช้เวลา 134.5 วินาที
ประมวลผล 400/2965 รูป | ใช้เวลา 201.1 วินาที
ประมวลผล 500/2965 รูป | ใช้เวลา 267.6 วินาที
ประมวลผล 600/2965 รูป | ใช้เวลา 337.8 วินาที
ประมวลผล 700/2965 รูป | ใช้เวลา 408.7 วินาที
ประมวลผล 800/2965 รูป | ใช้เวลา 477.1 วินาที
ประมวลผล 900/2965 รูป | ใช้เวลา 541.4 วินาที
ประมวลผล 1000/2965 รูป | ใช้เวลา 609.3 วินาที
ประมวลผล 1100/2965 รูป | ใช้เวลา 676.3 วินาที
ประมวลผล 1200/2965 รูป | ใช้เวลา 741.8 วินาที
ประมวลผล 1300/2965 รูป | ใช้เวลา 808.0 วินาที
ประมวลผล 1400/2965 รูป | ใช้เวลา 896.3 วินาที
ประมวลผล 1500/2965 รูป | ใช้เวลา 963.6 วินาที
ประมวลผล 1600/2965 รูป | ใช้เวลา 1031.5 วินาที
ประมวลผล 1700/2965 รูป | ใช้เวลา 1123.0 วินาที
ประมวลผล 1800/2965 รูป | ใช้เวลา 1190.1 วินาที
ประมวลผล 1900/2965 รูป | ใช้เวลา 1261.6 วินาที
ประมวลผล 2000/2965 รูป | ใช้เวลา 1326.4 วินาที
ประมวลผล 2100/2965 รูป | ใช้เวลา 1412.2 วินาที
ประมวลผล 2200/2965 รูป | ใช้เวลา 1497.5 

## โค้ดโมเดล + การโหลดข้อมูล
รายชื่อ backbone ทั้งหมดอยู่ใน `BACKBONES` — ยังไม่ได้เลือกตัวไหน ตัวที่ใช้จริงกำหนดตอนเรียก `train_model(backbone=...)`

In [ ]:
import csv
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
import keras

BACKBONES = {
    "densenet121":    (keras.applications.DenseNet121,      224, "torch"),
    "efficientnetb0": (keras.applications.EfficientNetB0,   224, "none"),
    "mobilenetv2":    (keras.applications.MobileNetV2,      224, "tf"),
    "inceptionv3":    (keras.applications.InceptionV3,      224, "tf"),
    "resnet50":       (keras.applications.ResNet50,         224, "caffe"),
    "efficientnetb3": (keras.applications.EfficientNetB3,   300, "none"),
    "efficientnetv2s": (keras.applications.EfficientNetV2S, 384, "none"),
}

# ปรับค่าตามโมเดล
@keras.saving.register_keras_serializable(package="gindaimai")
class BackbonePreprocess(keras.layers.Layer):
    def __init__(self, mode="none", **kwargs):
        super().__init__(**kwargs)
        self.mode = mode

    def call(self, image):
        if self.mode == "tf":
            return image / 127.5 - 1.0
        if self.mode == "torch":
           mean = tf.constant([0.485, 0.456, 0.406])
           std = tf.constant([0.229, 0.224, 0.225])
           image = image / 255.0
           return (image - mean) / std
        elif self.mode == "caffe":
          image = tf.reverse(image, axis=[-1])
          mean = tf.constant([103.939, 116.779, 123.68])
          return image - mean
        return image

    def get_config(self):
        config = super().get_config()
        config.update({"mode": self.mode})
        return config

def load_split(data_dir, split = None):
    data_dir = Path(data_dir)

    with open(data_dir / "classes.json", "r", encoding="utf-8") as file:
        classes = json.load(file)

    with open(data_dir / "dataset.csv", "r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    if split is not None:
        rows = [r for r in rows if r["split"] == split]

    paths = [
        str(data_dir / r["cache_path"])
        for r in rows
    ]

    labels = np.array([
        [float(r[name]) for name in classes]
        for r in rows
    ], dtype = np.float32)

    return paths, labels, rows, classes

def decode_image(path, size):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, [size, size])
    return img

def augment_image(img, size):
    big = int(size * 1.15)
    img = tf.image.resize(img, [big, big])
    img = tf.image.random_crop(img, [size, size, 3])
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 25.0)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    img = tf.image.random_hue(img, 0.02)
    return tf.clip_by_value(img, 0.0, 255.0)

def make_dataset(paths, labels, size, batch, training):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(len(paths))

    def process_image(path, label):
        image = decode_image(path, size)

        if training:
            image = augment_image(image, size)

        return image, label

    dataset = dataset.map(
        process_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.batch(batch)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

# คำนวณ Loss โดยเพิ่มน้ำหนักให้ส่วนผสมที่พบได้น้อย
@keras.saving.register_keras_serializable(package="gindaimai")
class WeightedBCE(keras.losses.Loss):

  def __init__(self, pos_weight=None, **kwargs):
    super().__init__(**kwargs)
    self.pos_weight = pos_weight

  def call(self, y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    weight = tf.constant(self.pos_weight, dtype=tf.float32)
    loss = -( weight * y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred) )

    return tf.reduce_mean(loss, axis=-1)

  def get_config(self):
    config = super().get_config()
    config.update({"pos_weight": self.pos_weight})

    return config

# สร้างโมเดลจำแนกส่วนผสมอาหาร
def build_model(backbone, n_classes, size, dropout=0.3, weights="imagenet"):
    constructor, _, mode = BACKBONES[backbone]
    base_model = constructor(
        include_top=False,
        weights=weights,
        input_shape=(size, size, 3),
        pooling="avg"
    )
    base_model.trainable = False
    image = keras.Input(shape=(size, size, 3))
    x = BackbonePreprocess(mode)(image)
    x = base_model(x, training=False)
    x = keras.layers.Dropout(dropout)(x)
    output = keras.layers.Dense(
        n_classes,
        activation="sigmoid"
    )(x)
    model = keras.Model(image, output)

    return model, base_model

# คำนวณน้ำหนักของแต่ละส่วนผสม
def pos_weights(labels):
    positive = labels.sum(axis=0).clip(min=1)
    negative = len(labels) - positive
    weight = np.sqrt(negative / positive)

    return np.clip(weight, 1.0, 10.0)

## โค้ดวัดผล
หา threshold ต่อส่วนผสมจาก val → วัด P/R/F1/AP บน test → baseline เดาจากชื่อเมนู

In [ ]:
import csv
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
from sklearn.metrics import average_precision_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_fscore_support

# หา threshold ที่ให้ F1-score ดีที่สุด
def tune_thresholds(Y, P):
    thresholds = []

    for i in range(Y.shape[1]):
        best_threshold = 0.5
        best_f1 = -1

        if Y[:, i].sum() < 3:
            thresholds.append(best_threshold)
            continue

        for threshold in np.arange(0.05, 0.96, 0.05):
            pred = (P[:, i] >= threshold).astype(int)
            score = f1_score(Y[:, i], pred, zero_division=0)

            if score > best_f1:
                best_f1 = score
                best_threshold = threshold

        thresholds.append(best_threshold)

    return np.array(thresholds)


# คำนวณ Average Precision
def get_ap(y, p):
    if y.sum() == 0:
        return np.nan

    return average_precision_score(y, p)


# ทำนายจากสูตรอาหารของเมนู
def dish_baseline(train_rows, test_rows, classes):
    ingredient_sum = defaultdict(lambda: np.zeros(len(classes)))
    menu_count = defaultdict(int)

    for row in train_rows:
        menu = row["variant"]

        for i, name in enumerate(classes):
            ingredient_sum[menu][i] += float(row[name])

        menu_count[menu] += 1

    predictions = []

    for row in test_rows:
        menu = row["variant"]
        count = menu_count[menu]

        if count == 0:
            predictions.append(np.zeros(len(classes)))
        else:
            predictions.append(ingredient_sum[menu] / count)

    return np.array(predictions)


# ให้โมเดลทำนายภาพ
def predict(model, paths, size, batch):
    labels = np.zeros((len(paths), 1), dtype=np.float32)

    dataset = make_dataset(
        paths, labels, size, batch, training=False
    )

    return model.predict(dataset, verbose=0)


# ประเมินผลโมเดล
def evaluate(model, data_dir, run_dir, size, batch=32):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    # อ่านข้อมูล
    train_paths, train_labels, train_rows, classes = load_split(
        data_dir, "train"
    )
    val_paths, Yv, _, _ = load_split(data_dir, "val")
    test_paths, Yt, test_rows, _ = load_split(data_dir, "test")

    # ทำนายผล
    Pv = predict(model, val_paths, size, batch)
    Pt = predict(model, test_paths, size, batch)

    # หา threshold จาก Validation
    thresholds = tune_thresholds(Yv, Pv)

    with open(run_dir / "thresholds.json", "w", encoding="utf-8") as f:
        json.dump(
            dict(zip(classes, thresholds.tolist())),
            f, indent=2, ensure_ascii=False
        )

    # แปลงค่าความน่าจะเป็นเป็น 0 หรือ 1
    pred = (Pt >= thresholds).astype(int)

    # คำนวณ Precision, Recall, F1 และจำนวนตัวอย่าง
    precision, recall, f1, support = precision_recall_fscore_support(
        Yt, pred, zero_division=0
    )

    ap = [get_ap(Yt[:, i], Pt[:, i]) for i in range(len(classes))]

    # บันทึกผลแยกตามส่วนผสม
    with open(run_dir / "test_per_class.csv", "w",
              newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([
            "ingredient", "threshold", "precision",
            "recall", "f1", "AP", "support"
        ])

        for i, name in enumerate(classes):
            writer.writerow([
                name, round(thresholds[i], 2),
                round(precision[i], 3),
                round(recall[i], 3),
                round(f1[i], 3),
                round(ap[i], 3) if not np.isnan(ap[i]) else "",
                int(support[i])
            ])

    # ประเมินผลแยกตามเมนูย่อย
    menu_images = defaultdict(list)

    for i, row in enumerate(test_rows):
        menu_images[row["variant"]].append(i)

    with open(run_dir / "test_per_variant.csv", "w",
              newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "variant", "n_images",
            "macro_f1_present_classes", "exact_match"
        ])

        for menu, indices in sorted(menu_images.items()):
            true = Yt[indices]
            predicted = pred[indices]

            present = true.sum(axis=0) > 0

            if present.any():
                macro_f1 = f1_score(
                    true[:, present], predicted[:, present],
                    average="macro", zero_division=0
                )
            else:
                macro_f1 = np.nan

            exact_match = (true == predicted).all(axis=1).mean()

            writer.writerow([
                menu, len(indices),
                round(macro_f1, 3) if not np.isnan(macro_f1) else "",
                round(exact_match, 3)
            ])

    # เปรียบเทียบกับการทำนายจากสูตรอาหาร
    baseline = dish_baseline(train_rows, test_rows, classes)
    valid = Yt.sum(axis=0) > 0

    summary = {
        "n_test_images": len(Yt),
        "macro_f1_tuned": float(f1_score(
            Yt[:, valid], pred[:, valid],
            average="macro", zero_division=0
        )),
        "micro_f1_tuned": float(f1_score(
            Yt, pred, average="micro", zero_division=0
        )),
        "macro_f1_at_0.5": float(f1_score(
            Yt[:, valid], (Pt >= 0.5)[:, valid],
            average="macro", zero_division=0
        )),
        "mAP": float(np.nanmean(ap)),
        "exact_match_ratio": float((Yt == pred).all(axis=1).mean()),
        "baseline_dish_prior_macro_f1": float(f1_score(
            Yt[:, valid], (baseline > 0.5)[:, valid],
            average="macro", zero_division=0
        )),
        "baseline_dish_prior_mAP": float(np.nanmean([
            get_ap(Yt[:, i], baseline[:, i])
            for i in range(len(classes))
        ])),
        "note": "Baseline knows the true dish variant."
    }

    with open(run_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, allow_nan=True)

    print(json.dumps(summary, indent=2, allow_nan=True))

    return summary

## ฟังก์ชันเทรน 1 โมเดล

In [ ]:
import time
import gc

def train_model(
    backbone,
    run_dir,
    data_dir,
    img_size = 224,
    batch = 32,
    epochs_head = 8,
    epochs_ft = 30,
    lr_head = 1e-3,
    lr_ft = 1e-4,
    dropout = 0.3,
    patience = 7,
    seed = 42,
    weights = "imagenet"):

    start_time = time.time()

    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)

    run = Path(run_dir)
    run.mkdir(parents=True, exist_ok=True)


    train_paths, Y_train, _, classes = load_split(data_dir, "train")
    val_paths, Y_val, _, _ = load_split(data_dir, "val")

    train_ds = make_dataset(
        train_paths, Y_train, img_size, batch, training=True
    )
    val_ds = make_dataset(
        val_paths, Y_val, img_size, batch, training=False
    )

    # คำนวณน้ำหนักของแต่ละ label
    weights = pos_weights(Y_train)
    loss_fn = WeightedBCE(weights.tolist())

    # สร้างโมเดล
    model, base = build_model(
        backbone, len(classes), img_size, dropout, weights="imagenet"
    )

    # บันทึกค่าที่ใช้ในการเทรน
    config = {
        "backbone": backbone,
        "img_size": img_size,
        "batch": batch,
        "epochs_head": epochs_head,
        "epochs_ft": epochs_ft,
        "lr_head": lr_head,
        "lr_ft": lr_ft,
        "dropout": dropout,
        "seed": seed,
        "classes": classes
    }

    with open(run / "config.json", "w", encoding="utf-8") as file:
        json.dump(config, file, indent=2, ensure_ascii=False)

    with open(run / "classes.json", "w", encoding="utf-8") as file:
        json.dump(classes, file, indent=2, ensure_ascii=False)

    # ตั้งค่า metric และการบันทึกโมเดล
    metrics = [
        keras.metrics.AUC(
            curve = "PR",
            multi_label = True,
            name = "pr_auc"
        )
    ]

    checkpoint = keras.callbacks.ModelCheckpoint(
        run / "model.keras",
        monitor = "val_pr_auc",
        mode = "max",
        save_best_only = True
    )

    logger = keras.callbacks.CSVLogger(
        run / "history.csv",
        append=True
    )

    # ขั้นตอนที่ 1: เทรนเฉพาะส่วนจำแนกผลลัพธ์
    print("Start train Head")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_head),
        loss=loss_fn,
        metrics=metrics
    )

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs_head,
        callbacks=[checkpoint, logger],
        verbose=2
    )

    # ขั้นตอนที่ 2: Fine-tune โมเดล
    print("Start Fine-tune")

    base.trainable = True

    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    steps = max(1, len(train_paths) // batch) * epochs_ft

    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=lr_ft,
        decay_steps=steps,
        alpha=0.05
    )

    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_schedule,
            weight_decay=1e-4
        ),
        loss=loss_fn,
        metrics=metrics
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_pr_auc",
        mode="max",
        patience=patience
    )

    model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=epochs_head,
        epochs=epochs_head + epochs_ft,
        callbacks=[checkpoint, logger, early_stop],
        verbose=2
    )

    # โหลดโมเดลที่ได้ผลดีที่สุด โดยส่ง custom_objects
    best_model = keras.models.load_model(
        run / "model.keras",
        custom_objects={
            "BackbonePreprocess": BackbonePreprocess,
            "WeightedBCE": WeightedBCE
        },
        compile=False
    )

    # ประเมินผลโมเดล
    summary = evaluate(
        best_model, data_dir, run, img_size, batch
    )

    # บันทึกเวลาในการเทรน
    total_time = time.time() - start_time

    with open(run / "train_seconds.txt", "w") as file:
        file.write(str(round(total_time)))

    # เคลียร์หน่วยความจำ
    del model, best_model
    gc.collect()
    keras.backend.clear_session()

    return summary

## 2. เตรียมข้อมูล

In [ ]:
if not os.path.exists(f"{DATA_DIR}/dataset.csv"):
  arg = {
      "labels" : ["labels_final_set1_1669.json", "labels_final_set2_1584.json"],
      "classes" : "classes_set1.json",
      "image_root" : IMAGE_ROOT,
      "out" : DATA_DIR,
      "min_pos" : 30,
      "duplicate_policy" : "lastest",
      "val" : 0.15,
      "test" : 0.15,
      "max_side" : 640,
      "seed" : 42,
      "verbose" : 1
  }

  prepare_dataset(argparse.Namespace(**args))

else:
    print("ข้อมูลเตรียมไว้แล้วใน session นี้ — ข้าม")

ข้อมูลเตรียมไว้แล้วใน session นี้ — ข้าม


## 3. เทรนและเปรียบเทียบ
เทรนทีละโมเดล ผลแต่ละตัวเก็บบน Drive ทันที — ถ้า Colab หลุด รัน cell นี้ใหม่ ตัวที่เสร็จแล้วจะถูกข้าม

In [ ]:
import keras
from pathlib import Path

keras.mixed_precision.set_global_policy("mixed_float16")

for name in MODELS:
  run = Path(COMPARE_DIR) / name

  # 1) เสร็จครบแล้ว (เทรน + ประเมินผล) → ข้าม
  if (run / "summary.json").exists():
    print(f"[complete] {name}")
    continue

  # 2) เทรนเสร็จแล้วแต่ยังไม่ได้ประเมินผล → โหลดโมเดลมาประเมินผล
  if (run / "model.keras").exists():
    print(f"[เทรนเสร็จแล้ว ประเมินผลอย่างเดียว] {name}")
    saved_model = keras.models.load_model(
        run / "model.keras",
        custom_objects={
            "BackbonePreprocess": BackbonePreprocess,
            "WeightedBCE": WeightedBCE
        },
        compile=False
    )
    evaluate(saved_model, DATA_DIR, run, IMG_SIZE, BATCH)
    continue

  # 3) ยังไม่เคยเทรน → เทรนใหม่
  train_model(
      name,
      run,
      DATA_DIR,
      IMG_SIZE,
      BATCH,
      EPOCHS_HEAD,
      EPOCHS_FT,
      seed=SEED)

[complete] densenet121
[complete] efficientnetb0
[complete] mobilenetv2
[complete] inceptionv3
[complete] resnet50


## 4. ผลการเปรียบเทียบ

In [ ]:
import json
import pandas as pd
from pathlib import Path

results = []

for name in MODELS:
    run = Path(COMPARE_DIR) / name
    file = run / "summary.json"

    if not file.exists():
        print("ยังไม่มีผล:", name)
        continue

    with open(file, "r", encoding="utf-8") as f:
        summary = json.load(f)

    exact_match = summary.get("exact_match_ratio", 0.0)
    macro_f1 = summary.get("macro_f1_tuned", 0.0)
    micro_f1 = summary.get("micro_f1_tuned", 0.0)
    macro_f1_05 = summary.get("macro_f1_at_0.5", 0.0)
    mAP = summary.get("mAP", 0.0)
    baseline_f1 = summary.get("baseline_dish_prior_macro_f1", 0.0)
    baseline_mAP = summary.get("baseline_dish_prior_mAP", 0.0)

    results.append([
        name, exact_match, macro_f1, micro_f1, macro_f1_05, mAP, baseline_f1, baseline_mAP
    ])

df = pd.DataFrame(
    results,
    columns=[
        "Model", "Exact Match (Acc)", "Macro-F1 (Tuned)", "Micro-F1 (Tuned)",
        "Macro-F1 (0.5)", "mAP", "Baseline F1", "Baseline mAP"
    ]
)

# เรียงตาม maP จากมากไปน้อย
df = df.sort_values(by="mAP", ascending=False)

# บันทึกผลลง CSV
output_file = Path(COMPARE_DIR) / "comparison.csv"
df.to_csv(output_file, index=False)

print("\n===== FINAL RESULT =====")
print(df.to_string(index=False))
print(f"\nบันทึกผลแล้วที่: {output_file}")


===== FINAL RESULT =====
         Model  Exact Match (Acc)  Macro-F1 (Tuned)  Micro-F1 (Tuned)  Macro-F1 (0.5)      mAP  Baseline F1  Baseline mAP
efficientnetb0           0.164009          0.679934          0.746922        0.698417 0.748358     0.731085      0.768865
      resnet50           0.184510          0.686705          0.754135        0.693821 0.746068     0.731085      0.768865
   densenet121           0.186788          0.682178          0.758395        0.696018 0.742114     0.731085      0.768865
   inceptionv3           0.161731          0.661845          0.727616        0.649787 0.729187     0.731085      0.768865
   mobilenetv2           0.125285          0.635544          0.735996        0.660374 0.706272     0.731085      0.768865

บันทึกผลแล้วที่: /content/drive/MyDrive/Colab Notebooks/ingredient_model/runs/compare/comparison.csv
